# DistilBERT fine-tune — Food Hazard Detection (SemEval-2025 Task 9, ST1)

Neural baseline for **Efialtis Stin Kouzina**. Two separate
DistilBERT models are trained: one for the `hazard-category` and one for the
`product-category`, fine-tuned on `title + text`.

Corresponds to section 12 of the report. The model is trained with plain
cross-entropy, **without class weighting** (as in the report) — which is why it
underperforms on the rare classes that macro-F1 cares about, and ultimately lost
to the TF-IDF + MiniLM stack.

## Instructions (Colab)

1. **Runtime → Change runtime type → GPU** (a free T4 is enough).
2. In the Files panel, upload `train.csv`, `valid.csv`, `test.csv` (from
   `data/raw/`). The notebook looks for them in the cwd or in `data/raw/`.
3. **Runtime → Run all**. Roughly ~30-45 minutes on a T4.
4. Download `submission_distilbert.csv` and submit it to Kaggle.

It also produces `logits_haz_*.npy`, `logits_prod_*.npy`, and `label_maps.json`
for a possible later ensemble with the TF-IDF stack (section 14).

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# configuration (same reasoning as section 12 of the report)
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256             # truncation - most titles+texts fit comfortably
BATCH = 16               # batch size for the fine-tune
EPOCHS = 4               # 4 epochs (the report notes 6-8 might have helped)
LR = 2e-5                # the classic learning rate for BERT fine-tuning
SEED = 42                # for reproducibility

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: without a GPU the fine-tune is very slow. Runtime -> GPU.")

In [ ]:
# load the data. train/valid carry an extra index column, test does not
# (same as src/io_utils.py in the local pipeline).
def find_csv(name):
    for cand in (Path(name + ".csv"), Path("data/raw") / (name + ".csv")):
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Upload {name}.csv (into the cwd or data/raw/).")


def load_split(name):
    if name in ("train", "valid"):
        return pd.read_csv(find_csv(name), index_col=0)
    return pd.read_csv(find_csv(name))


train = load_split("train")
valid = load_split("valid")
test = load_split("test")
print(f"train={len(train)}  valid={len(valid)}  test={len(test)}")


def build_text(df):
    # join title + text. No metadata here (BERT is left to see only the
    # text, unlike TF-IDF which also takes the metadata tokens).
    title = df.get("title", "").fillna("").astype(str)
    text = df.get("text", "").fillna("").astype(str)
    return (title + " " + text).str.strip()


for df in (train, valid, test):
    df["input_text"] = build_text(df)

In [ ]:
# build the label maps. Taken from train+valid pooled together so they cover
# every class (some rare class may be missing from train alone).
pooled = pd.concat([train, valid], ignore_index=True)


def build_label_maps(col):
    classes = sorted(pooled[col].unique().tolist())
    label2id = {c: i for i, c in enumerate(classes)}
    id2label = {i: c for c, i in label2id.items()}
    return label2id, id2label


haz_l2i, haz_i2l = build_label_maps("hazard-category")
prod_l2i, prod_i2l = build_label_maps("product-category")
print(f"hazard classes={len(haz_l2i)}  product classes={len(prod_l2i)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)  # dynamic padding per batch


# a simple torch Dataset that tokenizes the texts.
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels) if labels is not None else None

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

In [ ]:
# helpers for training and prediction.
def train_model(train_df, target_col, label2id, id2label, tag):
    # fine-tune one DistilBERT on one of the two tasks.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    ds = TextDataset(train_df["input_text"], train_df[target_col].map(label2id))
    args = TrainingArguments(
        output_dir=f"out_{tag}",
        per_device_train_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=torch.cuda.is_available(),   # fp16 on GPU for speed
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=ds,
        data_collator=collator, tokenizer=tokenizer,
    )
    trainer.train()
    return trainer


def predict_logits(trainer, texts):
    # return the raw logits (for argmax and for a possible ensemble later).
    ds = TextDataset(texts)
    return trainer.predict(ds).predictions


def score_st1(y_haz_true, y_haz_pred, y_prod_true, y_prod_pred):
    # Official ST1: (macroF1(hazard) + macroF1(product | hazard correct)) / 2
    f1_h = f1_score(y_haz_true, y_haz_pred, average="macro", zero_division=0)
    mask = np.asarray(y_haz_true) == np.asarray(y_haz_pred)
    if mask.sum() == 0:
        f1_p = 0.0
    else:
        f1_p = f1_score(np.asarray(y_prod_true)[mask], np.asarray(y_prod_pred)[mask],
                        average="macro", zero_division=0)
    return (f1_h + f1_p) / 2.0, f1_h, f1_p

In [ ]:
# step (1): train on train, evaluate on valid (single split).
haz_trainer = train_model(train, "hazard-category", haz_l2i, haz_i2l, "haz")
prod_trainer = train_model(train, "product-category", prod_l2i, prod_i2l, "prod")

haz_va_logits = predict_logits(haz_trainer, valid["input_text"])
prod_va_logits = predict_logits(prod_trainer, valid["input_text"])

haz_va_pred = np.array([haz_i2l[i] for i in haz_va_logits.argmax(1)])
prod_va_pred = np.array([prod_i2l[i] for i in prod_va_logits.argmax(1)])

st1, f1_h, f1_p = score_st1(
    valid["hazard-category"].to_numpy(), haz_va_pred,
    valid["product-category"].to_numpy(), prod_va_pred,
)
print(f"\n[valid] ST1={st1:.4f}  hazard_F1={f1_h:.4f}  product_F1(cond)={f1_p:.4f}")
# Caution: this is single-split and overestimates. See section 13 (cross-validation).

In [ ]:
# step (2): refit on train+valid and predict on test for the submission.
haz_full = train_model(pooled, "hazard-category", haz_l2i, haz_i2l, "haz_full")
prod_full = train_model(pooled, "product-category", prod_l2i, prod_i2l, "prod_full")

haz_te_logits = predict_logits(haz_full, test["input_text"])
prod_te_logits = predict_logits(prod_full, test["input_text"])

haz_te_pred = [haz_i2l[i] for i in haz_te_logits.argmax(1)]
prod_te_pred = [prod_i2l[i] for i in prod_te_logits.argmax(1)]

# write the Kaggle CSV with the proper columns.
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "hazard-category": haz_te_pred,
    "product-category": prod_te_pred,
})
submission.to_csv("submission_distilbert.csv", index=False)
print("wrote submission_distilbert.csv", submission.shape)
submission.head()

In [ ]:
# save the logits + label maps, so a soft-voting ensemble with the
# TF-IDF stack (section 14) can be built later without re-running BERT.
np.save("logits_haz_valid.npy", haz_va_logits)
np.save("logits_prod_valid.npy", prod_va_logits)
np.save("logits_haz_test.npy", haz_te_logits)
np.save("logits_prod_test.npy", prod_te_logits)

with open("label_maps.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "hazard": {str(i): c for i, c in haz_i2l.items()},
            "product": {str(i): c for i, c in prod_i2l.items()},
        },
        f, ensure_ascii=False, indent=2,
    )
print("saved logits_*.npy + label_maps.json")

## Notes

- The single-split valid ST1 here overestimates — see section 13 of the report
  (cross-validation). DistilBERT's final Kaggle score came in below the
  TF-IDF + MiniLM stack.
- For a stronger neural baseline: class weighting / focal loss on the rare
  classes, more epochs with early stopping, and trying DeBERTa-v3.
- The `logits_*.npy` files can go into a soft-voting ensemble with
  `notebooks/12_stacking_ensemble.py`.